# EDA dữ liệu nền (dữ liệu thô)

**Mục tiêu:** rà soát chất lượng bộ dữ liệu nền `data/interim/base/` trước khi làm sạch. Notebook này **chỉ đọc và mô tả, không sửa dữ liệu**.

**Câu hỏi cần trả lời**

1. Bảng panel có đủ 63 tỉnh × 10 năm không?
2. Các biến chính có giá trị, mức độ phân tán và khác biệt giữa các vùng ra sao (thống kê mô tả)?
3. Biến nào thiếu số liệu, thiếu ở năm nào?
4. Các cờ chất lượng (số sơ bộ, gián đoạn chuỗi) tập trung ở đâu?
5. Có giá trị vô lý hoặc mâu thuẫn giữa các chỉ tiêu không?
6. Có bước nhảy hoặc đỉnh bất thường giữa các năm không?

**Đầu ra:** bảng tổng hợp vấn đề ở mục 9, làm căn cứ cho `scripts/03_clean.py`.

**Đồng bộ với notebook 02:** notebook này và `02_eda_processed.ipynb` dùng chung các hàm trong `src/eda.py` và có cùng cấu trúc mục 1–8; chỉ khác dữ liệu đầu vào (`load("base")` / `load("clean")`).

## 1. Thiết lập và đọc dữ liệu

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Tìm thư mục gốc dự án (thư mục chứa src/), dù notebook được mở từ đâu
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

from src import eda

d = eda.load("base")   # dữ liệu nền: data/interim/base/

## 2. Cấu trúc panel

In [ ]:
eda.structure(d)

In [ ]:
# Độ phủ của toàn bộ chỉ tiêu trong indicators.csv
eda.indicator_coverage(d)

**Nhận xét**

- Panel **cân bằng**: 63 tỉnh × 10 năm = 630 dòng, không trùng lặp.
- Chỉ có **63 đơn vị**, là cỡ mẫu nhỏ đối với causal forest. Cần lưu ý khi diễn giải và nên có mô hình cơ sở (hồi quy hiệu ứng cố định) để so sánh.
- Phần lớn trong 1.458 chỉ tiêu chỉ có ở một vài năm hoặc một vài tỉnh, nên panel chỉ chọn 75 chỉ tiêu có độ phủ tốt. Các chỉ tiêu khác vẫn lấy được từ bảng observations khi cần.

## 3. Thống kê mô tả

Thống kê trên toàn bộ các quan sát tỉnh × năm của 16 biến chính, chia theo 5 nhóm: giá trị (3.1–3.3), tương quan giữa các biến (3.4) và biến thiên giữa các tỉnh và trong tỉnh (3.5). Danh sách biến nằm ở `eda.KEY_VARS` trong `src/eda.py`; khi nhóm chốt Y, W, X thì bổ sung các biến đó vào danh sách.

In [ ]:
# 3.1 Toàn bộ giai đoạn 2015–2024
eda.describe(d)

In [ ]:
# 3.2 Trung bình theo giai đoạn, ứng với hai ấn phẩm nguồn
eda.describe_by_period(d)

In [ ]:
# 3.3 Trung bình theo vùng (mã vùng ở mục 2)
eda.describe_by_region(d)

**Nhận xét**

- **Số quan sát:** phần lớn biến đủ 630 quan sát. Thu nhập bình quân và tỷ lệ hộ nghèo chỉ có khoảng 310–315 quan sát (giai đoạn 2020–2024).
- **Phân phối lệch phải mạnh** (skew > 2) ở dân số, mật độ dân số, GRDP bình quân đầu người, năng suất lao động và vốn FDI/GRDP (skew 6,2). Nguyên nhân là Hà Nội, TP. Hồ Chí Minh và một số tỉnh công nghiệp vượt trội. Khi đưa vào mô hình, các biến này nên lấy log.
- **Học sinh trên một giáo viên** có giá trị lớn nhất 70,1 và skew 3,9, bất thường với một tỷ lệ thường dao động 15–25. Mục 7 cho thấy nguyên nhân là lỗi số học sinh năm 2018.
- **Theo giai đoạn:** GRDP bình quân đầu người (giá hiện hành) tăng 47%, tỷ lệ lao động qua đào tạo tăng 27%, vốn FDI/GRDP giảm 21%.
- **Theo vùng:** chênh lệch rất lớn. GRDP bình quân đầu người ở Đông Nam Bộ (SE) gấp gần 3 lần Trung du miền núi phía Bắc (NMM); tỷ lệ hộ nghèo ở NMM 15,5% so với 0,7% ở SE. Mức độ không đồng nhất cao như vậy là điều kiện phù hợp để causal forest ước lượng tác động khác nhau giữa các tỉnh.

### 3.4 Ma trận tương quan

Hệ số tương quan r giữa từng cặp biến chính (từ −1 đến +1; |r| ≥ 0,7 coi là mạnh). Tính theo hai cách:

- **Pearson:** đo quan hệ tuyến tính, dễ bị vài giá trị cực đoan (Hà Nội, TP. Hồ Chí Minh) chi phối.
- **Spearman:** tính trên thứ hạng của các quan sát nên ít bị ngoại lai chi phối, phù hợp với các biến lệch phải mạnh như ở mục 3.1.

Tương quan chỉ cho biết hai biến có đi cùng nhau hay không, **không phải quan hệ nhân quả**.

In [ ]:
# Pearson: bản đồ nhiệt và các cặp |r| ≥ 0,7
eda.correlation(d)

In [ ]:
# Spearman: chỉ liệt kê các cặp |r| ≥ 0,7
eda.correlation(d, method="spearman", plot=False)

**Nhận xét**

- **GRDP bình quân đầu người và năng suất lao động** có r = 0,96 theo cả hai cách, gần như đo cùng một thứ. Khi chọn biến kiểm soát (X) chỉ nên đưa một trong hai vào mô hình.
- Theo Pearson chỉ có 4 cặp |r| ≥ 0,7; theo Spearman có 8 cặp. Spearman cho thấy một **cụm biến "mức độ phát triển"** liên quan chặt với nhau: GRDP bình quân đầu người, năng suất lao động, thu nhập bình quân, tỷ lệ hộ nghèo (r = −0,92 với thu nhập), mật độ dân số. Pearson đánh giá thấp các quan hệ này vì phân phối lệch. Khi chọn X nên xem cả hai cách và tránh đưa nhiều biến cùng cụm vào mô hình.
- **Học sinh trên một giáo viên và thu nhập bình quân** (Pearson r = 0,71): các tỉnh thu nhập cao, đô thị hóa cao có lớp học đông hơn. Đây là tương quan, không có nghĩa thu nhập làm tăng sĩ số lớp.
- Các cặp biến còn lại có |r| < 0,7, chưa có dấu hiệu trùng lặp nghiêm trọng giữa các nhóm biến.

### 3.5 Biến thiên giữa các tỉnh và trong từng tỉnh

Với dữ liệu panel, sự khác nhau của một biến đến từ hai nguồn:

- **Giữa các tỉnh:** tỉnh này luôn cao hơn tỉnh kia (ví dụ Hà Nội luôn đông dân hơn Bắc Kạn).
- **Trong từng tỉnh:** cùng một tỉnh nhưng thay đổi qua các năm.

Mô hình có **hiệu ứng cố định tỉnh và năm** chỉ học được từ phần biến thiên trong tỉnh sau khi bỏ xu hướng chung của cả nước theo năm (cột cuối của bảng). Biến nào có phần này quá nhỏ thì mô hình gần như không có thông tin để ước lượng tác động của biến đó. Cách tính tương tự lệnh `xtsum` của Stata.

In [ ]:
eda.variation(d)

**Nhận xét**

- **Gần như không đổi theo thời gian trong mỗi tỉnh:** dân số (0,4% phương sai nằm trong tỉnh), mật độ dân số (2%), tỷ lệ đô thị hóa (2,6%), tỷ lệ hộ nghèo (6,6%). Sau khi bỏ hiệu ứng cố định tỉnh và năm, các biến này chỉ còn dưới 5% biến thiên. Nếu dùng làm biến tác động (W) trong mô hình có hiệu ứng cố định tỉnh thì gần như không ước lượng được. Chúng phù hợp hơn để làm **đặc điểm của tỉnh** (X), hoặc làm biến để xem tác động khác nhau giữa các nhóm tỉnh.
- **Biến thiên nhiều trong tỉnh:** chỉ số phát triển GRDP (74%), vốn FDI/GRDP (49%), tỷ lệ thất nghiệp (47%), vốn đầu tư/GRDP (40%). Sau khi bỏ hiệu ứng cố định tỉnh và năm vẫn còn khoảng 38–57%. Đây là các ứng viên phù hợp cho Y hoặc W.
- **GRDP bình quân đầu người:** 16,5% biến thiên nằm trong tỉnh, nhưng chỉ còn 3,9% sau khi bỏ hiệu ứng năm, vì phần lớn thay đổi qua các năm là xu hướng tăng chung của cả nước.
- **Học sinh trên một giáo viên** có 47% biến thiên trong tỉnh, cao bất thường với một biến cơ cấu. Phần lớn là do lỗi số học sinh năm 2018 (mục 7); xem lại ở notebook 02 sau khi sửa.

## 4. Thiếu dữ liệu

In [ ]:
# Tỷ lệ ô trống theo biến
missing = eda.missing_by_var(d)

In [ ]:
# Số tỉnh thiếu theo biến × năm
eda.missing_heatmap(d, missing.index)

**Nhận xét.** Có hai kiểu thiếu dữ liệu, cần xử lý khác nhau:

| Kiểu thiếu | Biến | Nguyên nhân | Hướng xử lý |
|---|---|---|---|
| **Thiếu có cấu trúc** (cả năm thiếu ở mọi tỉnh hoặc phần lớn tỉnh) | `income_per_capita_month`, `poverty_rate` (chỉ có 2020–2024); nhóm doanh nghiệp (không có 2024); `firms_new` (không có 2015); `grdp_real_growth` (năm 2015 không có năm trước); `preschool_students` (2019 thiếu 44 tỉnh); `life_expectancy` (2015–2019 thiếu 13–17 tỉnh) | Ấn phẩm không công bố | **Không điền.** Nếu dùng các biến này thì giới hạn giai đoạn phân tích cho phù hợp |
| **Thiếu rải rác** (vài tỉnh, vài năm) | ngân sách (7 tỉnh, 2022–2024), vốn FDI thực hiện, thuê bao internet, số dự án FDI… | Tỉnh không báo cáo, hoặc ô "-", "…" trong ấn phẩm | Quyết định ở bước chuẩn bị dữ liệu mô hình, sau khi đã chốt biến |

## 5. Cờ chất lượng

In [ ]:
# 5.1 Số quan sát là số sơ bộ theo năm
eda.preliminary_by_year(d)

In [ ]:
# 5.2 Cờ series_break (chuỗi gián đoạn so với năm trước)
break_count = eda.breaks(d)

**Nhận xét**

- **Số sơ bộ** chủ yếu rơi vào năm 2024 (năm cuối của ấn phẩm mới nhất). Đề xuất kiểm định độ vững bằng cách **bỏ năm 2024**.
- **Năm 2020 cả 63 tỉnh đều có cờ gián đoạn**, vì là năm chuyển từ ấn phẩm 2015–2021 sang ấn phẩm 2020–2024. Mô hình cần có hiệu ứng cố định năm hoặc biến `post2020`.
- Cột `has_break` tính trên cả 75 biến nên rất rộng; khi lọc chỉ nên xét đúng các biến của mô hình (cột `break_vars`).
- Nhóm FDI bị gắn cờ nhiều nhất, phù hợp với đặc điểm biến động mạnh của dòng vốn FDI.

## 6. Kiểm tra tính hợp lý

In [ ]:
# 6.1 Giá trị âm
eda.negative_values(d)

In [ ]:
# 6.2 Biến tỷ lệ % nằm ngoài khoảng 0–100
eda.pct_out_of_range(d)

In [ ]:
# 6.3 Ràng buộc logic giữa các chỉ tiêu (số dòng vi phạm, kèm danh sách)
eda.logic_summary(d)

In [ ]:
# 6.4 Xem kỹ các tỉnh có thành phần GRDP lệch tổng ở dữ liệu nền
cols = ["province_name", "year", "grdp_current"] + eda.GRDP_PARTS
d["panel"].query("province_name in ['Vinh Long', 'Ha Nam', 'Bac Giang'] and (year <= 2018 or year >= 2023)")[cols]

**Nhận xét**

- **Giá trị âm** ở `firm_profit` (doanh nghiệp lỗ), `grdp_real_growth` (tăng trưởng âm) và `fdi_registered_usd` (điều chỉnh giảm vốn đăng ký) là **hợp lý**, giữ nguyên.
- **Tỷ lệ tiêm chủng > 100%** ở 2 dòng (Sơn La 2020: 100,1%; Bạc Liêu 2023: 101,8%), thường do số trẻ được tiêm vượt số trẻ ước tính làm mẫu số. Giữ nguyên.
- **Vĩnh Long 2024:** GRDP dịch vụ = 332.060,1 tỷ đồng, gấp khoảng 10 lần năm 2023 và gần bằng 4 lần tổng GRDP của tỉnh. Nghi **thừa một chữ số**, cần đối chiếu PDF.
- **Hà Nam 2015–2018** (lệch 6–8%) và **Bắc Giang 2016** (lệch 1,2%): tổng thành phần GRDP lớn hơn tổng GRDP. Ở các dòng này 3 khu vực đã cộng đủ tổng GRDP, vậy phần lệch nằm ở thuế sản phẩm. Cần đối chiếu PDF xem số thuế lấy từ ấn phẩm nào.
- **Lao động 3 khu vực lệch tổng** ở Tây Ninh 2019, Đồng Tháp 2022. Cần đối chiếu PDF.
- **Tăng trưởng GRDP tự tính lệch chỉ số GSO** ở 13 dòng, đúng các trường hợp đã gắn cờ quy tắc (D) trong dữ liệu nền. Khi cần tốc độ tăng trưởng, dùng `grdp_growth_index`.

## 7. Bước nhảy và đỉnh bất thường giữa các năm

Hai cách phát hiện bổ sung cho nhau:

- **Bước nhảy:** giá trị năm t gấp ≥ 5 lần hoặc chỉ còn ≤ 1/5 năm t−1. Bắt được các thay đổi rất lớn.
- **Đỉnh một năm:** với các biến quy mô (số người, số cơ sở, số thuê bao…), giá trị năm t lệch hơn 25% so với trung bình năm t−1 và t+1, trong khi hai năm đó gần nhau. Bắt được các lỗi nhỏ hơn mà cách thứ nhất bỏ sót.

In [ ]:
# 7.1 Bước nhảy ≥ 5 lần
eda.jumps(d)

In [ ]:
# 7.2 Đỉnh một năm ở các biến quy mô
eda.spikes(d)

In [ ]:
# 7.3 Ví dụ: học sinh trên một giáo viên (đường đỏ: TP. Hồ Chí Minh)
eda.lines(d, "students_per_teacher", "Học sinh trên một giáo viên", highlight="Ho Chi Minh City")

In [ ]:
# 7.4 Ví dụ: thuê bao internet trên 100 dân (đường đỏ: Yên Bái)
eda.lines(d, "internet_per_100", "Thuê bao internet trên 100 dân", highlight="Yen Bai")

**Nhận xét**

- **Nhóm FDI** nhảy nhiều nhất. Đây chủ yếu là **đặc tính thật**: một dự án lớn có thể làm vốn của tỉnh tăng hoặc giảm nhiều lần. Nếu dùng làm biến tác động (W) nên cân nhắc lấy log, lấy trung bình nhiều năm hoặc dùng tỷ lệ trên GRDP. Lợi nhuận doanh nghiệp, tỷ suất nhập cư và tỷ lệ thiếu việc làm dao động mạnh vì đổi dấu hoặc giá trị gốc gần 0, ít đáng lo.
- **Đỉnh một năm** phát hiện 3 nhóm **lỗi thật**, đã đối chiếu PDF:
    - **Số học sinh năm 2018** ở TP. Hồ Chí Minh và 8 tỉnh Đồng bằng sông Cửu Long: ấn phẩm 2015–2021 in lệch số giữa các tỉnh (ví dụ số của Kiên Giang 197.184 chính là số của Cần Thơ ở ấn phẩm cũ). Lỗi này làm học sinh trên một giáo viên của TP. Hồ Chí Minh năm 2018 lên 70 (biểu đồ 7.3).
    - **Gia Lai 2019:** thuê bao cố định (1.323) và di động (33) bị đổi chỗ.
    - **Số trường năm 2023** của Điện Biên (5.173) và Bình Phước (6.442), trong khi các năm lân cận khoảng 265–300.
- Các đỉnh còn lại (lao động theo khu vực, hợp tác xã, thuê bao cố định đang giảm dần, học sinh mầm non giảm năm 2021 do dịch COVID-19) **có thể là biến động thật**, chưa đủ căn cứ để coi là lỗi.
- **Thuê bao internet:** nhiều tỉnh nhảy 5–8 lần trong 2018–2022; Yên Bái có hơn 200 thuê bao/100 dân giai đoạn 2015–2019 rồi giảm mạnh năm 2020 (biểu đồ 7.4). Khả năng cao **phạm vi thống kê khác nhau giữa các tỉnh** trước năm 2020. Không nên dùng biến này nếu chưa xử lý được điểm này.

## 8. Diễn biến theo thời gian và ngoại lai

In [ ]:
# Mỗi đường mảnh là một tỉnh, đường đen là trung vị 63 tỉnh
eda.trends(d, ["grdp_pc", "invest_to_grdp", "invest_fdi_to_grdp",
               "budget_rev_domestic_to_grdp", "trained_labor_rate", "urbanization_rate"])

In [ ]:
# Các giá trị cao nhất: lỗi hay thực tế?
eda.top_values(d, ["invest_to_grdp", "invest_fdi_to_grdp", "budget_rev_domestic_to_grdp"])

**Nhận xét**

- **Ranh giới năm 2020:** tỷ lệ đô thị hóa có bước tăng bậc ngay năm 2020, đúng năm chuyển ấn phẩm (đồng thời là năm áp dụng kết quả Tổng điều tra dân số 2019); tỷ lệ lao động qua đào tạo tăng nhanh bất thường giai đoạn 2019–2021. Đây là gián đoạn do nguồn số liệu, hiệu ứng cố định năm sẽ hấp thụ phần lớn.
- **Vốn đầu tư/GRDP > 100%** ở Hà Tĩnh, Thái Nguyên, Thanh Hóa giai đoạn 2015–2016 trùng với thời kỳ triển khai các dự án FDI rất lớn tại các tỉnh này (khu liên hợp gang thép, tổ hợp điện tử, lọc hóa dầu). Đây là **số liệu thật**, giữ nguyên, nhưng cần kiểm định độ vững khi bỏ các quan sát này.
- **Hà Nội, TP. Hồ Chí Minh** vượt trội ở hầu hết các biến quy mô. Đây là ngoại lai thật, không xóa.

## 9. Tổng hợp vấn đề và đề xuất cho bước làm sạch

| # | Vấn đề | Biến, phạm vi | Loại | Đề xuất |
|---|---|---|---|---|
| 1 | Nghi lệch một chữ số | Vĩnh Long 2024, `grdp_current_serv` | Lỗi | Đối chiếu PDF; sửa theo đẳng thức GRDP = 3 khu vực + thuế |
| 2 | Thành phần GRDP lệch tổng | Hà Nam 2015–2018, Bắc Giang 2016 | Lỗi ghép nguồn | Đối chiếu PDF; xác định nguồn của số thuế sản phẩm |
| 3 | Số học sinh in lệch giữa các tỉnh | 2018: TP. HCM và 8 tỉnh ĐBSCL | Lỗi ấn phẩm | Đối chiếu PDF; dùng số ấn phẩm 2015–2018 nếu hợp lý, nếu không thì để trống |
| 4 | Đổi chỗ hai chỉ tiêu | Gia Lai 2019, thuê bao cố định ↔ di động | Lỗi ấn phẩm | Đổi lại chỗ |
| 5 | Số trường bất thường | Điện Biên, Bình Phước 2023 | Lỗi ấn phẩm | Để trống |
| 6 | Lao động 3 khu vực lệch tổng | Tây Ninh 2019, Đồng Tháp 2022 | Mâu thuẫn | Đối chiếu PDF |
| 7 | GRDP giá so sánh không khớp chỉ số phát triển | 13 dòng | Mâu thuẫn trong ấn phẩm | Không sửa; dùng `grdp_growth_index` |
| 8 | Phạm vi thống kê khác nhau | Thuê bao internet trước 2020 | Gián đoạn chuỗi | Không dùng, hoặc chỉ dùng 2020–2024 |
| 9 | Gián đoạn tại năm 2020 | Đô thị hóa, lao động qua đào tạo, ngân sách… | Gián đoạn chuỗi | Giữ; mô hình dùng hiệu ứng cố định năm hoặc `post2020` |
| 10 | Biến động rất mạnh | Nhóm FDI | Đặc tính thật | Giữ; ưu tiên `invest_fdi`; cân nhắc log hoặc trung bình nhiều năm |
| 11 | Thiếu có cấu trúc | Thu nhập, hộ nghèo, doanh nghiệp 2024, mầm non 2019, tuổi thọ 2015–2019 | Thiếu | Không điền; chọn giai đoạn phù hợp |
| 12 | Thiếu rải rác | Ngân sách, FDI thực hiện, internet… | Thiếu | Xử lý ở bước dữ liệu mô hình |
| 13 | Ngoại lai thật, phân phối lệch | Hà Nội, TP. HCM; Hà Tĩnh, Thái Nguyên, Thanh Hóa 2015–2016 | Đặc tính thật | Giữ; lấy log biến quy mô; kiểm định độ vững |
| 14 | Số sơ bộ | Chủ yếu năm 2024 | Chất lượng | Giữ; kiểm định độ vững bỏ năm 2024 |

**Kết luận.** Dữ liệu nền **dùng được**: panel cân bằng, đa số biến có độ phủ trên 95%. Lỗi thật sự (mục 1–5) chỉ ở vài chục ô và đều có thể kiểm chứng bằng PDF. Phần lớn các "bất thường" còn lại là đặc tính thật của dữ liệu hoặc gián đoạn do nguồn, nên xử lý bằng **thiết kế mô hình** (hiệu ứng cố định năm, lấy log, kiểm định độ vững) chứ không sửa số. Kết quả xử lý: xem `notebooks/02_eda_processed.ipynb`.